# Performance comparison "Ollama" vs "llama.cpp"
In this notebook we will use Nvidia GenAI-Perf to benchmark performance of Ollama vs llama.cpp. To do a fair comparison we are going to choose `llama3.1:8b` which is available on both platforms with the same quantization `Q4_K_M`

## Preliminary steps
### Install Ollama and pull the Llama3.1:8b model

```
curl -fsSL https://ollama.com/install.sh | sh
systemctl status ollama
ollama pull llama3.1:8b
ollama run llama3.1:8b
```
Run a simple query before the test to ensure the memory is loaded when the test starts

### Install llama.cpp

```
sudo apt update
sudo apt install -y cmake build-essential git
git clone https://github.com/ggml-org/llama.cpp
cd llama.cpp/
/usr/local/cuda/bin/nvcc --version
export CUDACXX=/usr/local/cuda/bin/nvcc
cmake --version
cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j$(nproc)
```


### Get the GGUF model file from Huggingface
```
source .venv/bin/activate
pip install -U huggingface-hub
hf download bartowski/Meta-Llama-3.1-8B-Instruct-GGUF \
  --include "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
  --local-dir .
./build/bin/llama-server \
  -m ./Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf \
  -ngl 99 \
  --ctx-size 4096 \
  --port 8080
curl http://localhost:8080/v1/models | python3 -m json.tool
```


## GenAI-Perf

### Install GenAI-Perf and log in to HF to ensure tokenizer is accessible
```
python3 -m venv aiperf
source aiperf/bin/activate
python -m pip install --upgrade pip
pip install aiperf
aiperf --version
hf auth login
```

## Run basic test against both models
### Llama.cpp
```
aiperf profile \
  --model "llama" \
  --streaming \
  --endpoint-type chat \
  --tokenizer meta-llama/Llama-3.1-8B-Instruct \
  --url http://localhost:8080/v1 \
  --concurrency 5 \
  --isl 100 \
  --request-count 10 \
  --profile-export-file in100con5
```
### Ollama
```
aiperf profile \
  --model "llama3.1:8b" \
  --streaming \
  --endpoint-type chat \
  --tokenizer meta-llama/Llama-3.1-8B-Instruct \
  --url http://localhost:11434 \
  --concurrency 5 \
  --isl 100 \
  --request-count 10 \
  --profile-export-file in100con5
```

## Show Ollama results

In [14]:
import pandas as pd
df = pd.read_csv("/home/demouser/Desktop/aiperf/artifacts/llama3.1:8b-openai-chat-concurrency5/in100con5.csv", nrows=26)
df.drop(columns=["sum", "p1", "p5", "p10", "p25", "p50", "p75", "p95"], inplace=True)
df = df[~df["Metric"].astype(str).str.contains("HTTP", na=False)]
df

,Metric,avg,min,max,p90,p99,std
0,E2E Output Token Throughput (tokens/sec/user),74.62,45.83,111.21,110.55,111.15,29.44
15,Input Sequence Length (tokens),100.00,100.00,100.00,100.00,100.00,0.00
16,Inter Chunk Latency (ms),8.69,5.88,22.82,8.94,9.66,0.95
17,Inter Token Latency (ms),8.70,8.08,8.92,8.90,8.92,0.31
18,Output Sequence Length (tokens),244.20,184.00,321.00,296.20,318.52,48.25
19,Output Token Count (tokens),244.20,184.00,321.00,296.20,318.52,48.25
20,Output Token Throughput Per User (tokens/sec/u...,115.14,112.13,123.79,119.98,123.41,4.38
21,Prefill Throughput Per User (tokens/sec/user),693.10,28.04,2059.63,1750.15,2028.68,836.55
22,Request Latency (ms),3785.00,1679.48,5650.84,5464.86,5632.24,1466.53
23,Time to First Output Token (ms),1671.34,48.55,3566.73,3313.57,3541.42,1439.63


## Show Llama.cpp results

In [15]:
df2 = pd.read_csv("/home/demouser/Desktop/aiperf/artifacts/llama-openai-chat-concurrency5/in100con5.csv", nrows=26)
df2.drop(columns=["sum", "p1", "p5", "p10", "p25", "p50", "p75", "p95"], inplace=True)
df2 = df2[~df2["Metric"].astype(str).str.contains("HTTP", na=False)]
df2

,Metric,avg,min,max,p90,p99,std
0,E2E Output Token Throughput (tokens/sec/user),110.08,59.06,133.71,128.39,133.17,26.13
15,Input Sequence Length (tokens),100.00,100.00,100.00,100.00,100.00,0.00
16,Inter Chunk Latency (ms),7.49,3.02,274.53,6.97,11.53,13.92
17,Inter Token Latency (ms),7.34,5.55,8.10,8.08,8.10,0.98
18,Output Sequence Length (tokens),221.60,157.00,260.00,256.00,259.60,37.79
19,Output Token Count (tokens),221.60,157.00,260.00,256.00,259.60,37.79
20,Output Token Throughput Per User (tokens/sec/u...,139.09,123.43,180.16,164.92,178.64,21.65
21,Prefill Throughput Per User (tokens/sec/user),847.31,55.79,1052.48,1050.09,1052.24,395.79
22,Request Latency (ms),2090.13,1510.78,2658.18,2458.51,2638.21,365.36
23,Time to First Output Token (ms),435.00,95.01,1792.28,1113.92,1724.45,678.64


## Conclusion
Ollama offers convenience but performs worse
- TTFT: 1671 ms vs 435 ms
- Tk/s: 74 vs 110